In [ ]:
import sys
import os

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

import polars as pl
from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline
from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory

In [ ]:
# 2. Run pipeline
config = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://bmlldata',
        start_date='2024-01-02',
        end_date='2024-01-02',
        exchanges=['ARCX'],
        data_types=['trades']
    ),
    processing=ProcessingConfig(
        normalization=BMLLNormalizer()
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://bmlldata'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir},
                  flat_core_count=5,
                  memory_multiplier=2.0,
                  memory_per_core_gb=4.0,
                  max_retries=5,
                  cpu_buffer=1,
                  file_sort_order="desc")
)

pipeline = Pipeline(config)
results = pipeline.run(slice(10))

In [ ]:
# 3. Verify results
import pandas as pd
res_pd = pd.DataFrame(results)
successful = res_pd.query("message == 'success'").shape[0]
failed = res_pd.query("message != 'success'").shape[0]
print(f"Successful: {successful}, Failed: {failed}")

In [ ]:
# 4. Inspect paths
result = results[0]
print(f"Input:  {result['input_path']}")
print(f"Output: {result['output_path']}")
print(f"Rows:   {result['row_count']:,}")

In [ ]:
# 5. Read output
data_access = DataAccessFactory.create('s3', region='us-east-1')
output_data = data_access.read(result['output_path']).limit(100).collect()
print(f"✓ Read {len(output_data)} records")
output_data.head()

In [ ]:
# 6. Validate schema
normalizer = BMLLNormalizer()
expected_schema = normalizer.get_schema('trades')
missing = set(expected_schema.keys()) - set(output_data.columns)
assert not missing, f"Missing columns: {missing}"
print(f"✓ Schema validated ({len(output_data.columns)} columns)")

In [ ]:
# 7. Compare raw input files with normalized output files
import pandas as pd

raw_files = pipeline._discover_files()
norm_data_access = DataAccessFactory.create('s3', region='us-east-1')
norm_files = config.processing.normalization.discover_files(
    norm_data_access,
    config.storage.normalized.path,
    config.ray.file_sort_order
)

def extract_file_id(path):
    parts = path.split('/')
    if len(parts) >= 6:
        return '/'.join(parts[-6:])
    return path.split('/')[-1]

raw_dict = {extract_file_id(f[0]): {'path': f[0], 'size_gb': f[1]} for f in raw_files}
norm_dict = {extract_file_id(f[0]): {'path': f[0], 'size_gb': f[1]} for f in norm_files}

comparison_data = []
for file_id in set(raw_dict.keys()) & set(norm_dict.keys()):
    comparison_data.append({
        'file_id': file_id,
        'raw_size_gb': raw_dict[file_id]['size_gb'],
        'norm_size_gb': norm_dict[file_id]['size_gb'],
        'size_ratio': norm_dict[file_id]['size_gb'] / raw_dict[file_id]['size_gb'] if raw_dict[file_id]['size_gb'] > 0 else 0,
        'raw_path': raw_dict[file_id]['path'],
        'norm_path': norm_dict[file_id]['path']
    })

comparison_df = pd.DataFrame(comparison_data)
print(f"Avg size ratio: {comparison_df['size_ratio'].mean():.3f}")
display(comparison_df.head(20))

In [ ]:
# 8. Compare dataframes for files with unusual size ratios
unusual = comparison_df[((comparison_df['size_ratio'] < 0.8) | (comparison_df['size_ratio'] > 1.2)) & (comparison_df['raw_size_gb'] > 0.01)].copy()
unusual['data_type'] = unusual['file_id'].str.split('/').str[-3]
trades = unusual[unusual['data_type'] == 'trades'].head(5)
l2 = unusual[unusual['data_type'] == 'level2q'].head(5)

print(f"Found {len(unusual)} files with unusual size ratios")
print(f"Trades: {len(trades)}, L2: {len(l2)}")

for idx, row in pd.concat([trades, l2]).iterrows():
    print(f"\n{'='*80}")
    print(f"File: {row['file_id']}")
    print(f"Size ratio: {row['size_ratio']:.3f}")
    
    raw_df = data_access.read(row['raw_path']).collect()
    norm_df = data_access.read(row['norm_path']).collect()
    
    # Column comparison
    raw_cols = set(raw_df.columns)
    norm_cols = set(norm_df.columns)
    print(f"\nColumns only in raw: {raw_cols - norm_cols}")
    print(f"Columns only in norm: {norm_cols - raw_cols}")
    print(f"Common columns: {len(raw_cols & norm_cols)}")
    
    # Nanosecond timestamp outer join - different columns per data type
    if 'trades' in row['file_id']:
        ns_cols = ['TradeTimestampNanoseconds', 'PublicationTimestampNanoseconds']
    else:  # level2q
        ns_cols = ['TimestampNanoseconds']
    
    ts_col = next((col for col in ns_cols if col in raw_cols and col in norm_cols), None)
    if ts_col:
        # Multi-column join for precise matching
        join_cols = [ts_col]
        for col in ['ExchangeTicker', 'Ticker', 'ISOExchangeCode', 'MIC', 'OPOL', 'InstrumentId', 'ExchangeSequenceNo', 'BMLLSequenceNo', 'BMLLSequenceSource']:
            if col in raw_cols and col in norm_cols:
                join_cols.append(col)
        
        raw_ts = raw_df.select(join_cols).unique()
        norm_ts = norm_df.select(join_cols).unique()
        only_raw = raw_ts.join(norm_ts, on=join_cols, how='anti')
        only_norm = norm_ts.join(raw_ts, on=join_cols, how='anti')
        print(f"\nNanosecond timestamps only in raw: {len(only_raw)}")
        print(f"Nanosecond timestamps only in norm: {len(only_norm)}")
        if len(only_raw) > 0:
            print(f"Sample raw-only ns timestamps: {only_raw.head(5).to_pandas()[ts_col].tolist()}")
        if len(only_norm) > 0:
            print(f"Sample norm-only ns timestamps: {only_norm.head(5).to_pandas()[ts_col].tolist()}")
    
    # Price comparison
    price_col = None
    for col in ['Price', 'price', 'TradePrice', 'BidPrice']:
        if col in raw_cols and col in norm_cols:
            price_col = col
            break
    
    if price_col and ts_col:
        joined = raw_df.select(join_cols + [price_col]).join(
            norm_df.select(join_cols + [price_col]),
            on=join_cols,
            how='inner',
            suffix='_norm'
        )
        if len(joined) > 0:
            diff = joined.with_columns(
                (pl.col(price_col) - pl.col(f"{price_col}_norm")).alias('price_diff')
            )
            mismatches = diff.filter(pl.col('price_diff').abs() > 0.0001)
            print(f"\nPrice column '{price_col}' comparison:")
            print(f"  Matching records: {len(joined)}")
            print(f"  Price mismatches: {len(mismatches)}")
            if len(mismatches) > 0:
                print(f"  Sample mismatches:")
                display(mismatches.head(5))
    
    print(f"\nRaw shape: {raw_df.shape}, Norm shape: {norm_df.shape}")

# Add specific file comparison
specific_norm_path = 's3://orderflowanalysis/intermediate/normalized/2024/07/05/level2q/AMERICAS/XCIS-20240705.parquet'
try:
    specific_norm_df = data_access.read(specific_norm_path).collect()
    print(f"\n{'='*80}")
    print(f"Specific file: {specific_norm_path}")
    print(f"Shape: {specific_norm_df.shape}")
    print(f"Columns: {list(specific_norm_df.columns)}")
except Exception as e:
    print(f"\nCould not read specific file: {e}")
    # Column comparison
    raw_cols = set(raw_df.columns)
    norm_cols = set(norm_df.columns)
    print(f"\nColumns only in raw: {raw_cols - norm_cols}")
    print(f"Columns only in norm: {norm_cols - raw_cols}")
    print(f"Common columns: {len(raw_cols & norm_cols)}")
    
    # Nanosecond timestamp outer join - different columns per data type
    if 'trades' in row['file_id']:
        ns_cols = ['TradeTimestampNanoseconds', 'PublicationTimestampNanoseconds']
    else:  # level2q
        ns_cols = ['TimestampNanoseconds']
    
    ts_col = next((col for col in ns_cols if col in raw_cols and col in norm_cols), None)
    if ts_col:
        # Multi-column join for precise matching
        join_cols = [ts_col]
        for col in ['ExchangeTicker', 'Ticker', 'ISOExchangeCode', 'MIC', 'OPOL', 'InstrumentId', 'ExchangeSequenceNo', 'BMLLSequenceNo', 'BMLLSequenceSource']:
            if col in raw_cols and col in norm_cols:
                join_cols.append(col)
        
        raw_ts = raw_df.select(join_cols).unique()
        norm_ts = norm_df.select(join_cols).unique()
        only_raw = raw_ts.join(norm_ts, on=join_cols, how='anti')
        only_norm = norm_ts.join(raw_ts, on=join_cols, how='anti')
        print(f"\nNanosecond timestamps only in raw: {len(only_raw)}")
        print(f"Nanosecond timestamps only in norm: {len(only_norm)}")
        if len(only_raw) > 0:
            print(f"Sample raw-only ns timestamps: {only_raw.head(5).to_pandas()[ts_col].tolist()}")
        if len(only_norm) > 0:
            print(f"Sample norm-only ns timestamps: {only_norm.head(5).to_pandas()[ts_col].tolist()}")
    
    # Price comparison
    price_col = None
    for col in ['Price', 'price', 'TradePrice', 'BidPrice']:
        if col in raw_cols and col in norm_cols:
            price_col = col
            break
    
    if price_col and ts_col:
        joined = raw_df.select(join_cols + [price_col]).join(
            norm_df.select(join_cols + [price_col]),
            on=join_cols,
            how='inner',
            suffix='_norm'
        )
        if len(joined) > 0:
            diff = joined.with_columns(
                (pl.col(price_col) - pl.col(f"{price_col}_norm")).alias('price_diff')
            )
            mismatches = diff.filter(pl.col('price_diff').abs() > 0.0001)
            print(f"\nPrice column '{price_col}' comparison:")
            print(f"  Matching records: {len(joined)}")
            print(f"  Price mismatches: {len(mismatches)}")
            if len(mismatches) > 0:
                print(f"  Sample mismatches:")
                display(mismatches.head(5))
    
    print(f"\nRaw shape: {raw_df.shape}, Norm shape: {norm_df.shape}")